# Preprocessing

This notebook prepares the cleaned visits for a later model. It does not train a model, and it does not change the 15.63% buyer share.

The order is fixed: load, split, then fit any transform on the training rows only.


### A. Load the cleaned table

`read_csv` turns `OperatingSystems`, `Browser`, `Region`, and `TrafficType` back into numbers. They are category codes, so they are cast to text before anything else. The checks confirm the cleaning notebook's result: 12,205 rows and no missing cells.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

np.random.seed(42)

if Path("data/processed/cleaned_shoppers.csv").exists():
    processed_dir = Path("data/processed")
    image_dir = Path("images")
    model_dir = Path("models")
else:
    processed_dir = Path("../data/processed")
    image_dir = Path("../images")
    model_dir = Path("../models")

image_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(processed_dir / "cleaned_shoppers.csv")
code_columns = ["OperatingSystems", "Browser", "Region", "TrafficType"]
for column in code_columns:
    df[column] = df[column].astype(str)

assert len(df) == 12205
assert int(df.isna().sum().sum()) == 0
print("Rows:", len(df))
print("Missing cells:", int(df.isna().sum().sum()))
print(df.dtypes.to_string())


Rows: 12205
Missing cells: 0
Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                       object
OperatingSystems            object
Browser                     object
Region                      object
TrafficType                 object
VisitorType                 object
Weekend                      int64
Revenue                      int64


### B. Split before any fitting

The rows are split 80/20 before any scaling or encoding. Stratified on `Revenue` means the training part and the test part each keep about the same buyer share. `random_state=42` makes this same split every time the notebook is run.

The files saved here are still the original numbers. Nothing has been scaled yet.


In [2]:
from sklearn.model_selection import train_test_split

feature_columns = [column for column in df.columns if column != "Revenue"]
X = df[feature_columns]
y = df["Revenue"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)


def report_split(name, y_part):
    buyers = int(y_part.sum())
    print(f"{name}: {len(y_part)} rows, {buyers} buyers, {100 * buyers / len(y_part):.2f}%")


report_split("Train", y_train)
report_split("Test", y_test)

original_columns = df.columns.tolist()
train_clean = X_train.copy()
train_clean["Revenue"] = y_train
test_clean = X_test.copy()
test_clean["Revenue"] = y_test
train_clean = train_clean[original_columns]
test_clean = test_clean[original_columns]

train_path = processed_dir / "train_clean.csv"
test_path = processed_dir / "test_clean.csv"
train_clean.to_csv(train_path, index=False)
test_clean.to_csv(test_path, index=False)
print("Saved", train_path)
print("Saved", test_path)


Train: 9764 rows, 1526 buyers, 15.63%
Test: 2441 rows, 382 buyers, 15.65%
Saved ../data/processed/train_clean.csv
Saved ../data/processed/test_clean.csv
